# 05 — Integration test: train, approve and predict

Run All after deploying with notebook 02. For each selected use case, this test:

1. Starts preliminary training in parallel, then verifies its metadata and registered model.
2. Starts full training using each preliminary run's exact metadata URI.
3. Verifies those full-model versions and approves **both** model approval and
   **Development stage status** on those exact versions.
4. Runs `prediction_test` and `prediction_inference` in parallel for each full model.
5. Verifies prediction/analysis files and checks that scoring loaded the training metadata.

Two models produce **eight pipeline executions**. This is an integration test of
the normal Processing path, not a model-quality assessment or every possible
configuration. HPO, native Batch Transform, drift comparisons and MLflow finalization
are outside this first matrix. Cache hits are reported and remain enabled.

The run cell starts billable SageMaker jobs and changes the two approval fields
on this test's verified full-model versions. It does not deploy pipelines, approve
preliminary models or select an unrelated latest model. Outputs and approved test
models are retained for inspection. Use your own development namespace.

Allow roughly an hour based on your workloads; the report measures actual time.
The notebook kernel must remain running to launch later phases. If disconnected,
keep the same RUN_LABEL and inputs to resume saved executions. A new label starts
a new test. A failed phase stops progression; jobs already running are not cancelled.

## Load the deployed application

- Locate this checkout and read the namespace/publication saved by notebook 02.
- Import the integration helper and ordinary AWS SDK; no model libraries are loaded.
- Prepare the display utilities without starting jobs or changing approvals.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import HTML, display
from html import escape

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
for directory in (ROOT / "notebooks/source/05_integration_test", ROOT / "mlops-models/src"):
    sys.path.insert(0, str(directory))
from integration_checks import IntegrationSuite

config_path = ROOT / "notebooks/config.json"
config = json.loads(config_path.read_text()) if config_path.is_file() else {}
namespace = config.get("namespace", "")
publication_path = ROOT / ".runs/source-deploy" / namespace / "publication.json"
publication = json.loads(publication_path.read_text()) if namespace and publication_path.is_file() else {}
print("Namespace:", namespace or "set NAMESPACE below")
print("Publication:", publication_path if publication else "provide PIPELINES below")

## Choose the test inputs

- Keep a unique test label; reuse it and the same inputs to resume after a kernel restart.
- Leave MODEL_IDS empty to test every deployed ModelId, or list the two you want.
- Use deployed CSV defaults, or supply per-model training/test/prediction CSVs below.

`prediction_test` needs the labelled data expected by your analysis code. Omitted
inference CSVs reuse that model's `training` override, if supplied; otherwise they
use the published inference default. CSVs remain in their existing S3 locations.
For a Feature Store deployment, leave CSV_INPUTS empty to use its published bindings.

PIPELINES can be left empty with a saved publication, or set to
`{"training": "<pipeline name or ARN>", "inference": "<pipeline name or ARN>"}`.
The first inspection pins versions; resuming this test keeps those saved versions.

In [ ]:
RUN_LABEL = "integration-001"
MODEL_IDS = []
PIPELINES = {}
NAMESPACE = namespace
AWS_REGION = publication.get("region", "")
MODEL_CODE_SOURCE = "artifact"  # "pipeline" tests the published inference code instead.
WAIT_TIMEOUT_MINUTES = 120
CSV_INPUTS = {
    # "usecase1": {
    #     "training": "s3://your-bucket/labelled-usecase1.csv",
    #     "prediction_test": "s3://your-bucket/labelled-usecase1.csv",
    #     "prediction_inference": "s3://your-bucket/scoring-usecase1.csv",
    # },
    # "usecase2": {"training": "s3://your-bucket/labelled-usecase2.csv"},
}

## Inspect the matrix before starting

- Read both deployed definitions, pin their versions and validate the selected paths.
- Confirm that both belong to this namespace and use the Development registry workflow.
- Save the test configuration and show the planned executions; no jobs start here.

In [ ]:
suite = IntegrationSuite(
    ROOT, publication, run_label=RUN_LABEL, model_ids=MODEL_IDS,
    csv_inputs=CSV_INPUTS, pipelines=PIPELINES, namespace=NAMESPACE, region=AWS_REGION,
    model_code_source=MODEL_CODE_SOURCE, timeout_minutes=WAIT_TIMEOUT_MINUTES)
plan = suite.prepare()
for kind, pipeline in suite.pipelines.items():
    print(kind, pipeline["arn"], "version", pipeline["version"])
print("Planned executions:", len(plan))
print("Evidence:", suite.directory)
display(pd.DataFrame(plan))

## Run the integration test

- Start each parallel phase, wait for completion and verify its required artifacts.
- Approve both fields on the verified full models, then run both inference modes.
- Refresh progress and save a final check table even when a phase fails.

**Effect:** this cell launches jobs and approves the exact full models created by
this test. The role needs pipeline execution, job inspection, S3 read/list and model
package update permissions. The 120-minute setting bounds waiting, not job runtime.
Failures and timeouts leave existing jobs/resources available for investigation.

In [ ]:
progress = display(HTML("Starting integration test…"), display_id=True)

def show_progress(report):
    columns = ["model", "phase", "status", "checks", "active_steps", "minutes", "cached_steps"]
    rows = pd.DataFrame(report["executions"], columns=columns)
    progress.update(HTML(
        f"<p><b>{escape(report['status'])}</b> · {report['elapsed_minutes']:.1f} minutes · "
        f"{report['passed_checks']} checks passed · {report['failed_checks']} failed</p>"
        + rows.to_html(index=False)))

suite.on_update = show_progress
try:
    suite.run()
finally:
    result = suite.report()
    print("Result:", result["status"], "| elapsed minutes:", result["elapsed_minutes"])
    if result["error"]:
        print("Failure:", result["error"])
    display(pd.DataFrame(result["checks"]))
    print("Saved evidence:", suite.directory / "results.json")

## Inspect executions and outputs

- Display the exact execution ARNs, registered model versions and S3 locations.
- Inspect the saved results after completion or a failure without starting another job.
- Use notebook 04 with an execution ARN for individual step and job inspection.

A passed result means the selected normal paths produced the required files and
passed the recorded identity/metadata checks. It does not establish predictive
quality, quota headroom or the behavior of configurations outside this matrix.
Changing code and redeploying requires a new RUN_LABEL to test the new versions.

In [ ]:
result = suite.report()
display(pd.DataFrame(result["executions"]))
print("Checks:", result["passed_checks"], "passed,", result["failed_checks"], "failed")
print("State for resume:", suite.path)